In [1]:
import json
import sys
import re
from rich import print as rp
from collections import Counter
from pathlib import Path
from datetime import datetime

nb_dir = Path.cwd()
project_root = nb_dir.parent.parent
sys.path.insert(0, str(project_root))

from scripts.text_matching import normalise_text

In [2]:
folder_reparsed = Path(project_root / "data_reload/reparse_missing/reparsed")
folder_prepping = Path(project_root / "data_reload/reparse_missing/prepping")

topics_db_file = Path(project_root / "data_reload/db_exports/topics-2026-08-07.json")

full_file = folder_prepping / "missing_reparsed_full.json"
people_file = folder_prepping / "missing_reparsed_people.json"
references_file = folder_prepping / "missing_references.json"
no_authors_file = folder_prepping / "missing_no_authors.json"

In [3]:
with open(topics_db_file, "r") as f:
    topics = json.load(f)

counts = Counter()
parsed_data = {}
people_dict = {}
references = {}
no_authors = {}

topic_norm_lookup = {topic["topic_normalised"]: topic["topic_id"] for topic in topics}

for file in folder_reparsed.iterdir():

    with open(file, "r") as f:
       entries = json.load(f)

    for entry in entries:
        counts["entries"] += 1
        composite_id = entry["custom_id"]
        cid_parts = composite_id.split("_")
        topic_normalised = cid_parts[0]
        # rp(topic_normalised)

        if "price" in entry:
            price = entry["price"]
        else:
            price = None
            counts["missing_price"] += 1

        parsed = entry["parsed_entry"]

        title = parsed["title"]
        subtitle = parsed["subtitle"]
        publisher = parsed["publisher"]
        place_of_publication = parsed["place_of_publication"]
        publication_year = parsed["publication_year"]
        edition = parsed["edition"]
        pages = parsed["pages"]

        format_original = parsed["format_original"]
        format_expanded = parsed["format_expanded"]
        condition = parsed["condition"]
        copies = parsed["copies"]
        illustrations = parsed["illustrations"]
        packaging = parsed["packaging"]
        is_translation = parsed["is_translation"]
        original_language = parsed["original_language"]
        is_multivolume = parsed["is_multivolume"]
        volumes = parsed["volumes"]
        series_title = parsed["series_title"]
        total_volumes = parsed["total_volumes"]
        original_entry_pipes = parsed["administrative"]["original_entry"]
        is_reference = parsed["administrative"]["is_reference"]
        corrected_by_api = parsed["administrative"]["corrected_by_api"]
        missing_person = parsed["administrative"]["missing_person"]
        multiple_editions = parsed["administrative"]["multiple_editions"]
        api_concerned = parsed["administrative"]["api_concerned"]
        problematic_multi_volume = parsed["administrative"]["problematic_multi_volume"]
        verification_notes = parsed["administrative"].get("verification_notes", "")

        topic_id = topic_norm_lookup[topic_normalised]

        if is_reference == True:
            counts["references"] += 1
            references[composite_id] = entry
            continue

        # set is_active
        is_active = 1

        if api_concerned:
            is_active = 3
        elif corrected_by_api or missing_person or multiple_editions or verification_notes:
            is_active = 2
        elif problematic_multi_volume:
            is_active = 4

        # imported price
        if price:
            imported_price = True
        else:
            imported_price = False

        # format original entry
        original_entry = original_entry_pipes.replace(" || ", "\n")

        # people (extract first, so admin_data can record has_author)
        authors = parsed.get("authors") or []
        editors = parsed.get("editors") or []
        contributors = parsed.get("contributors") or []
        translator = parsed["translator"]

        book_people = []

        has_author = False

        if authors:
            has_author = True
        else:
            has_author = False
            counts["no_authors"] += 1
            no_authors[composite_id] = parsed

        # book data
        parsed_data[composite_id] = [
            {
                "books_data": {
                    "composite_id": composite_id,
                    "is_active": is_active,
                    "title": title,
                    "subtitle": subtitle,
                    "publisher": publisher,
                    "place_of_publication": place_of_publication,
                    "publication_year": publication_year,
                    "edition": edition,
                    "pages": pages,
                    "format_original": format_original,
                    "format_expanded": format_expanded,
                    "condition": condition,
                    "copies": copies,
                    "illustrations": illustrations,
                    "packaging": packaging,
                    "topic_id": topic_id,
                    "is_translation": is_translation,
                    "original_language": original_language,
                    "is_multivolume": is_multivolume,
                    "series_title": series_title,
                    "total_volumes": total_volumes
                },

                "admin_data": {
                    "composite_id": composite_id,
                    "original_entry": original_entry,
                    "corrected_by_api": corrected_by_api,
                    "missing_person": missing_person,
                    "multiple_editions": multiple_editions,
                    "api_concerned": api_concerned,
                    "problematic_multi_volume": problematic_multi_volume,
                    "verification_notes": verification_notes,
                    "has_author": has_author,
                },

                "price_data": {
                    "amount": price,
                    "imported_price": imported_price
                }
            }
        ]

        for sort_order, person in enumerate(authors, start=1):
            display_name = person["display_name"]
            name_normalised = normalise_text(display_name)

            person_record = {
                "display_name": display_name,
                "composite_id": composite_id,
                "sort_order": sort_order,
                "is_author": True,
                "is_editor": False,
                "is_contributor": False,
                "is_translator": False
            }

            if name_normalised not in people_dict:
                people_dict[name_normalised] = []
            people_dict[name_normalised].append(person_record)

            book_people.append(person_record)

        for sort_order, person in enumerate(editors, start=1):
            display_name = person["display_name"]
            name_normalised = normalise_text(display_name)

            person_record = {
                "display_name": display_name,
                "composite_id": composite_id,
                "sort_order": sort_order,
                "is_author": False,
                "is_editor": True,
                "is_contributor": False,
                "is_translator": False
            }

            if name_normalised not in people_dict:
                people_dict[name_normalised] = []
            people_dict[name_normalised].append(person_record)

            book_people.append(person_record)

        for sort_order, person in enumerate(contributors, start=1):
            display_name = person["display_name"]
            name_normalised = normalise_text(display_name)

            person_record = {
                "display_name": display_name,
                "composite_id": composite_id,
                "sort_order": sort_order,
                "is_author": False,
                "is_editor": False,
                "is_contributor": True,
                "is_translator": False
            }

            if name_normalised not in people_dict:
                people_dict[name_normalised] = []
            people_dict[name_normalised].append(person_record)

            book_people.append(person_record)

        if translator:
            display_name = translator["display_name"]
            name_normalised = normalise_text(display_name)

            person_record = {
                "display_name": display_name,
                "composite_id": composite_id,
                "sort_order": 1,
                "is_author": False,
                "is_editor": False,
                "is_contributor": False,
                "is_translator": True
            }

            if name_normalised not in people_dict:
                people_dict[name_normalised] = []
            people_dict[name_normalised].append(person_record)

            book_people.append(person_record)

        parsed_data[composite_id][0]["people_data"] = book_people


with open(full_file, "w") as f:
    json.dump(parsed_data, f, ensure_ascii=False, indent=2)

with open(people_file, "w") as f:
    json.dump(people_dict, f, ensure_ascii=False, indent=2)

with open(references_file, "w") as f:
    json.dump(references, f, ensure_ascii=False, indent=2)

with open(no_authors_file, "w") as f:
    json.dump(no_authors, f, ensure_ascii=False, indent=2)


rp(f"""
=== LOG ===
entries seen: {counts["entries"]}
references skipped: {counts["references"]}
books written: {len(parsed_data)}
unique people: {len(people_dict)}
no authors: {counts["no_authors"]}
missing price: {counts["missing_price"]}

files written:
  {full_file.name}: {len(parsed_data)}
  {people_file.name}: {len(people_dict)}
  {references_file.name}: {len(references)}
  {no_authors_file.name}: {len(no_authors)}

=== DONE ===
""")

=== LOG ===
entries seen: 1008
references skipped: 40
books written: 968
unique people: 1110
no authors: 88
missing price: 0

files written:
  missing_reparsed_full.json: 968
  missing_reparsed_people.json: 1110
  missing_references.json: 40
  missing_no_authors.json: 88

=== DONE ===